# Phase 4: Cross-lingual transfer (MuRIL, hi -> mr / bn)

Owner: Priyadarshi Kumar (Lead). Plan: [docs/01-roadmap.md](../docs/01-roadmap.md), Phase 4.

Take the three Hindi MuRIL checkpoints from Phase 3 (n=300,000, seeds 42, 43,
and 44, pinned revisions) and:

1. **Zero-shot** evaluate each on Marathi and Bengali gold (no further training).
   Hindi gold is scored first, and the run stops unless each model is within
   0.001 of its Phase 3 score.
2. Fine-tune MuRIL **in-language** on 300,000 `mr` and 300,000 `bn` sentences
   with the Phase 3 recipe, once per seed, to get the in-language baseline and
   the transfer gap with its seed-to-seed spread.

**Gate:** transfer matrix filled (train `hi`, test `hi`/`mr`/`bn`) **plus**
in-language upper bounds for `mr` and `bn`.

**Expected finding** ([docs/05-pitfalls.md](../docs/05-pitfalls.md)): hi -> mr
(same Devanagari script) should transfer better than hi -> bn (different
script). Measure it; that gap is the research angle for the writeup.

**How to run (Kaggle)**
1. Commit and push this notebook to `phase4-cross-lingual`, because the run
   clones that branch and records its commit.
2. Import it on Kaggle with **GPU T4 x2** and **Internet** on, and attach the
   `GH_TOKEN` (read-only) and `HF_TOKEN` (write) Secrets.
3. With `DEBUG = True`, use **Save & Run All**. This checks the full pipeline on
   tiny slices. Do not keep its `results.csv`.
4. Set `DEBUG = False` and use **Save & Run All** again (about 6.7 hours).
5. Download `NaamRekha/results/results.csv`,
   `NaamRekha/results/phase4_transfer_matrix.png`, and the executed notebook.
   Do not edit their contents.
6. Check the gate with `uv run python scripts/phase4_gate.py`.

In [ ]:
# Pin versions (docs/04-compute.md), same pins as Phase 3.
%pip install -q transformers==5.22.1 datasets==5.1.0 accelerate==1.15.0 seqeval==1.2.2 sentencepiece==0.2.2

In [ ]:
import base64
import os
import subprocess
import sys

# Train on one GPU so per_device_train_batch_size is the real batch size.
# Kaggle's 2x T4 otherwise triggers DataParallel and silently doubles it.
# Must be set before torch is imported.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/iPriyadarshi/NaamRekha.git"
REPO_BRANCH = "phase4-cross-lingual"


def get_secret(name):
    # Kaggle Secret, then Colab Secret, then environment variable.
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


# Local: the notebook runs from notebooks/, so the repo root is "..".
# Colab/Kaggle: clone the repo next to the notebook. Check for a repo file, not
# just src/, because Kaggle has its own unrelated /kaggle/src folder.
if os.path.isfile("../src/data.py"):
    REPO_ROOT = os.path.abspath("..")
elif os.path.isfile("src/data.py"):
    REPO_ROOT = os.path.abspath(".")
else:
    REPO_ROOT = os.path.abspath("NaamRekha")
    if not os.path.isdir(REPO_ROOT):
        env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
        token = get_secret("GH_TOKEN")
        if token:
            # Passed through the environment so it is never saved in .git/config
            # (Kaggle keeps that folder as output) or printed if the clone fails.
            auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            env.update(GIT_CONFIG_COUNT="1",
                       GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                       GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT],
                       check=True, env=env)
sys.path.insert(0, REPO_ROOT)

hf_token = get_secret("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

GIT_COMMIT = subprocess.check_output(["git", "-C", REPO_ROOT, "rev-parse", "--short", "HEAD"], text=True).strip()

from src import LABELS, evaluate, id2label, label2id, load_split, log_result, tag_distribution

print("repo:", REPO_ROOT, "| commit:", GIT_COMMIT)
print("labels:", LABELS)


## Config

Every knob is in this one cell. The three Hindi checkpoints, working subset size,
and training recipe are the final Phase 3 values
([docs/phase3/README.md](../docs/phase3/README.md)).

In [ ]:
SEED = 42            # global default; each run below sets its own seed
SEEDS = [42, 43, 44]
DEBUG = True         # True: tiny slices, 1 epoch (pipeline check). False: real runs on Kaggle.

SOURCE_LANG = "hi"
TARGET_LANGS = ["mr", "bn"]
EVAL_LANGS = [SOURCE_LANG, *TARGET_LANGS]

MODEL_KEY = "MuRIL"
BASE_CHECKPOINT = "google/muril-base-cased"

# Phase 3 Hindi models, pinned to their exact uploaded revisions. Each zero-shot
# hi -> hi score must be within HI_REPRO_TOLERANCE of its logged Phase 3 gold F1.
HI_MODELS = {
    42: {"repo": "priyadarshi-su/naamrekha-muril_hi_n300000_e3_lr2e-05_s42_bfd254ed",
         "revision": "de5204680ed4a8ab1ac842847df6866a6e2c9251", "gold_f1": 0.8274137068534269},
    43: {"repo": "priyadarshi-su/naamrekha-muril_hi_n300000_e3_lr2e-05_s43_223b5780",
         "revision": "f15d615b7d760d23af6c5aa111675f0d6be39c3d", "gold_f1": 0.824173091270828},
    44: {"repo": "priyadarshi-su/naamrekha-muril_hi_n300000_e3_lr2e-05_s44_618532f6",
         "revision": "aa2f28136b49859a460217190b85ebedfc625e32", "gold_f1": 0.8174346201743462},
}
HI_REPRO_TOLERANCE = 0.001    # about two entities on the 1,929-entity Hindi gold set
HI_SUBSET_SIZE = 300_000
HI_EPOCHS = 3

# Phase 3 training recipe, used for the in-language runs.
TRAIN_SUBSET_SIZE = HI_SUBSET_SIZE
MAX_LENGTH = 128
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
LR = 2e-5
WARMUP_RATIO = 0.1
EPOCHS = 3

HF_PRIVATE = False    # visibility of the in-language checkpoint repos (DEBUG repos are always private)

if DEBUG:
    SEEDS = SEEDS[:2]    # two seeds are enough to exercise the per-seed loops
    TRAIN_SUBSET_SIZE = 80
    EPOCHS = 1
    BATCH_SIZE = 8


In [ ]:
import random

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device, "| visible GPUs:", torch.cuda.device_count())
if device.type == "cpu" and not DEBUG:
    print("WARNING: no GPU. Use Kaggle for the real runs.")
if not DEBUG and torch.cuda.device_count() != 1:
    raise RuntimeError(f"real runs need exactly one visible GPU, found {torch.cuda.device_count()}")


def pick_storage() -> str:
    # Persistent storage for checkpoints: Kaggle working dir, Google Drive, or local checkpoints/.
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working/naamrekha"
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        return "/content/drive/MyDrive/NaamRekha"
    except ImportError:
        return os.path.join(REPO_ROOT, "checkpoints")


STORAGE = pick_storage()
os.makedirs(os.path.join(STORAGE, "phase4"), exist_ok=True)
print("checkpoints ->", os.path.join(STORAGE, "phase4"))

In [ ]:
# Check the HF Hub login before any training, so a missing token fails here
# rather than after a multi-hour run.
from huggingface_hub import HfApi

hf_api = HfApi()
HF_NAMESPACE = hf_api.whoami()["name"]
print("HF Hub user:", HF_NAMESPACE)


## Data

Gold test sets for all three languages are always scored in full
([docs/03-conventions.md](../docs/03-conventions.md)). Silver train sets for
`mr`/`bn` are loaded only for the in-language runs.

In [ ]:
def entity_report(rows, name):
    counts = tag_distribution(rows)
    total = sum(counts.values())
    ents = {t: counts.get(f"B-{t}", 0) for t in ("PER", "ORG", "LOC")}
    with_ent = sum(any(t != "O" for t in tags) for _, tags in rows)
    print(f"{name}: {len(rows):,} sentences | entities {ents} | O = {counts['O'] / total:.1%} of tokens | "
          f"{with_ent / len(rows):.1%} of sentences have >=1 entity")


def make_subset(rows, n, seed=SEED):
    n = min(n, len(rows))
    idx = sorted(random.Random(seed).sample(range(len(rows)), n))
    return [rows[i] for i in idx]


gold = {lang: load_split(lang, "test") for lang in EVAL_LANGS}
for lang, rows in gold.items():
    entity_report(rows, f"{lang} gold test")

## Label alignment and prediction

Same code as Phase 3 ([docs/05-pitfalls.md](../docs/05-pitfalls.md)): each
word's label goes on its first real subword; other subwords and special tokens
get `-100`. Predictions are read from those same positions. Words lost to
truncation stay in the gold labels and count as predicted `O`. Every Phase 4
score comes from a freshly loaded model in full 32-bit precision, because the
object used for fp16 training keeps an autocast wrapper on its `forward`. The self-test
runs on a Marathi and a Bengali sentence, since those are the new scripts in
this phase.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer


def label_positions(word_ids, tokens):
    # word -> position of the subword that carries its label: the first piece,
    # skipping a standalone SentencePiece "▁" when a real piece follows it.
    positions = {}
    for pos, wid in enumerate(word_ids):
        if wid is not None and (wid not in positions or tokens[positions[wid]] == "▁"):
            positions[wid] = pos
    return positions


def tokenize_and_align_labels(batch, tokenizer, max_length=MAX_LENGTH):
    enc = tokenizer(batch["words"], is_split_into_words=True, truncation=True, max_length=max_length)
    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        word_ids = enc.word_ids(batch_index=i)
        labeled = set(label_positions(word_ids, enc.tokens(batch_index=i)).values())
        label_ids = []
        for pos, wid in enumerate(word_ids):
            if wid is None:
                label_ids.append(-100)                  # [CLS] / [SEP] / padding
            elif pos in labeled:
                label_ids.append(label2id[tags[wid]])   # labeled subword of this word
            else:
                label_ids.append(-100)                  # other subword
        all_labels.append(label_ids)
    enc["labels"] = all_labels
    return enc


def to_hf_dataset(rows, tokenizer, max_length=MAX_LENGTH):
    ds = Dataset.from_dict({"words": [r[0] for r in rows], "tags": [r[1] for r in rows]})
    return ds.map(lambda b: tokenize_and_align_labels(b, tokenizer, max_length),
                  batched=True, remove_columns=["words", "tags"])


def predict_tags(model, tokenizer, rows, max_length=MAX_LENGTH, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    dev = next(model.parameters()).device
    y_true, y_pred = [], []
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        enc = tokenizer([r[0] for r in chunk], is_split_into_words=True, truncation=True,
                        max_length=max_length, padding=True, return_tensors="pt").to(dev)
        with torch.no_grad():
            preds = model(**enc).logits.argmax(-1).cpu().numpy()
        for b, (_, tags) in enumerate(chunk):
            pred_tags = ["O"] * len(tags)
            for wid, pos in label_positions(enc.word_ids(batch_index=b), enc.tokens(batch_index=b)).items():
                pred_tags[wid] = id2label[int(preds[b][pos])]
            y_pred.append(pred_tags)
            y_true.append(list(tags))
    return y_true, y_pred


In [ ]:
_tok = AutoTokenizer.from_pretrained(BASE_CHECKPOINT)
for lang in TARGET_LANGS:
    words, tags = gold[lang][0]
    labels = to_hf_dataset([gold[lang][0]], _tok)[0]["labels"]
    enc = _tok(words, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH)
    word_ids = enc.word_ids()
    assert len(labels) == len(word_ids)
    labeled = set(label_positions(word_ids, enc.tokens()).values())
    for pos, wid in enumerate(word_ids):
        if wid is None or pos not in labeled:
            assert labels[pos] == -100, f"{lang}: special/other subword must be -100"
        else:
            assert labels[pos] == label2id[tags[wid]], f"{lang}: labeled subword must carry the word's tag"
    unk = sum(t == _tok.unk_token_id for t in _tok(words, is_split_into_words=True)["input_ids"])
    long = sum(len(_tok(w, is_split_into_words=True)["input_ids"]) > MAX_LENGTH for w, _ in gold[lang])
    print(f"{lang} label-alignment self-test: PASSED ({len(words)} words -> {len(word_ids)} subwords, "
          f"{unk} [UNK]) | gold sentences > {MAX_LENGTH} subwords: {long} / {len(gold[lang])}")


## Logging

One helper so every row carries the same notes, including the code commit and
library versions. `DEBUG` rows are marked so the gate ignores them.

In [ ]:
import transformers


def record(seed, train_lang, test_lang, subset_size, epochs, scores, checkpoint_path, notes):
    notes = (f"phase=4; seed={seed}; {notes}; git={GIT_COMMIT}; torch={torch.__version__}; "
             f"transformers={transformers.__version__}" + ("; DEBUG run -- not a real result" if DEBUG else ""))
    log_result(model=MODEL_KEY, train_lang=train_lang, test_lang=test_lang, subset_size=subset_size,
               epochs=epochs, scores=scores, checkpoint_path=checkpoint_path, notes=notes)
    with open(os.path.join(REPO_ROOT, "results", "results.csv"), encoding="utf-8") as f:
        print(f.read().splitlines()[-1])


## 1. Zero-shot transfer (train `hi`, test `hi` / `mr` / `bn`)

No training. For each seed, Hindi is scored first and must be within 0.001 of
that model's Phase 3 gold score. If it isn't, the run stops before logging that
seed's rows, because its `mr`/`bn` numbers couldn't be trusted either. Rows log
the score measured here, so a small difference from Phase 3 stays visible.
Phase 3 scored its model right after training, under fp16 autocast, so the
tolerance allows for a few near-tied predictions flipping in 32-bit.

In [ ]:
import gc

from transformers import AutoModelForTokenClassification

zero_shot = {}    # (seed, test language) -> scores
for seed in SEEDS:
    hi = HI_MODELS[seed]
    hi_tokenizer = AutoTokenizer.from_pretrained(hi["repo"], revision=hi["revision"])
    hi_model = AutoModelForTokenClassification.from_pretrained(hi["repo"], revision=hi["revision"]).to(device)
    assert hi_model.config.label2id == label2id, "checkpoint label map differs from src.label2id"
    hi_url = f"https://huggingface.co/{hi['repo']}/tree/{hi['revision']}"

    seed_scores = {}
    for lang in EVAL_LANGS:
        rows = gold[lang][:50] if DEBUG else gold[lang]
        y_true, y_pred = predict_tags(hi_model, hi_tokenizer, rows)
        seed_scores[lang] = evaluate(y_true, y_pred)
        if lang == SOURCE_LANG and not DEBUG and abs(seed_scores[lang]["F1_overall"] - hi["gold_f1"]) > HI_REPRO_TOLERANCE:
            raise RuntimeError(f"seed {seed}: hi -> hi scored {seed_scores[lang]['F1_overall']:.4f}, "
                               f"Phase 3 logged {hi['gold_f1']:.4f}")

    for lang, scores in seed_scores.items():
        zero_shot[(seed, lang)] = scores
        kind = "in-language (Phase 3 check)" if lang == SOURCE_LANG else "zero-shot"
        record(seed, SOURCE_LANG, lang, HI_SUBSET_SIZE, HI_EPOCHS, scores, hi_url,
               f"{kind}; checkpoint={hi['repo']}; revision={hi['revision']}")

    del hi_model
    gc.collect()
    torch.cuda.empty_cache()


## 2. In-language baselines (train `mr` -> test `mr`, train `bn` -> test `bn`)

Same recipe and subset size as the Hindi models, run once per seed, so the only
planned difference from the zero-shot rows is the training language. After
training, each model is reloaded from its saved files and scored in full 32-bit,
the same way as the zero-shot models. Each run uploads its final model,
confirms the upload, logs a link pinned to that revision, and only then deletes
the local copy. If the upload fails, the run stops and the local folder stays.
`run_name` contains a config hash, so a run can only resume from a checkpoint
made with identical settings.

In [ ]:
import gc
import hashlib
import json
import math
import shutil

from transformers import DataCollatorForTokenClassification, Trainer, TrainingArguments


def publish_checkpoint(ckpt_dir, run_name):
    repo_id = f"{HF_NAMESPACE}/naamrekha-{run_name}"
    hf_api.create_repo(repo_id, private=HF_PRIVATE or DEBUG, exist_ok=True)
    commit = hf_api.upload_folder(repo_id=repo_id, folder_path=ckpt_dir,
                                  ignore_patterns=["checkpoint-*", "runs/*"])
    remote = {f.rfilename: f.size for f in hf_api.model_info(repo_id, revision=commit.oid, files_metadata=True).siblings}
    local_size = os.path.getsize(os.path.join(ckpt_dir, "model.safetensors"))
    if remote.get("model.safetensors") != local_size:
        raise RuntimeError(f"upload to {repo_id} not confirmed; keeping {ckpt_dir}")
    return repo_id, commit.oid


def train_run(lang, subset_size, epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE, seed=SEED, resume=True):
    random.seed(seed)
    torch.manual_seed(seed)

    train = make_subset(load_split(lang, "train"), subset_size, seed)
    entity_report(train, f"{lang} train subset")

    tokenizer = AutoTokenizer.from_pretrained(BASE_CHECKPOINT)
    model = AutoModelForTokenClassification.from_pretrained(
        BASE_CHECKPOINT, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )
    train_ds = to_hf_dataset(train, tokenizer)

    # transformers 5 dropped warmup_ratio, so convert the ratio to a step count.
    n_gpu = max(torch.cuda.device_count(), 1)
    total_steps = math.ceil(len(train_ds) / (batch_size * n_gpu)) * epochs
    warmup_steps = int(WARMUP_RATIO * total_steps)

    config = {"checkpoint": BASE_CHECKPOINT, "lang": lang, "n": len(train), "epochs": epochs, "lr": lr,
              "batch_size": batch_size, "seed": seed, "max_length": MAX_LENGTH, "gpus": n_gpu,
              "warmup_steps": warmup_steps}
    config_hash = hashlib.sha1(json.dumps(config, sort_keys=True).encode()).hexdigest()[:8]
    run_name = (f"{MODEL_KEY.lower()}_{lang}_n{len(train)}_e{epochs}_lr{lr}_s{seed}_{config_hash}"
                + ("_debug" if DEBUG else ""))
    ckpt_dir = os.path.join(STORAGE, "phase4", run_name)
    has_ckpt = os.path.isdir(ckpt_dir) and any(f.startswith("checkpoint-") for f in os.listdir(ckpt_dir))

    args = TrainingArguments(
        output_dir=ckpt_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        warmup_steps=warmup_steps,
        fp16=torch.cuda.is_available(),
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=50,
        seed=seed,
        report_to=[],
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      data_collator=DataCollatorForTokenClassification(tokenizer), processing_class=tokenizer)
    trainer.train(resume_from_checkpoint=has_ckpt and resume)
    trainer.save_model(ckpt_dir)
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    # Score a fresh copy: the trained object still has fp16 autocast on forward,
    # while the zero-shot models run in full 32-bit.
    eval_tokenizer = AutoTokenizer.from_pretrained(ckpt_dir)
    eval_model = AutoModelForTokenClassification.from_pretrained(ckpt_dir).to(device)
    rows = gold[lang][:50] if DEBUG else gold[lang]
    y_true, y_pred = predict_tags(eval_model, eval_tokenizer, rows)
    scores = evaluate(y_true, y_pred)

    repo_id, revision = publish_checkpoint(ckpt_dir, run_name)
    checkpoint_url = f"https://huggingface.co/{repo_id}/tree/{revision}"

    record(seed, lang, lang, len(train), epochs, scores, checkpoint_url,
           f"in-language baseline; lr={lr}; bs={batch_size}; effective_bs={batch_size * n_gpu}; gpus={n_gpu}; "
           f"warmup_steps={warmup_steps}; epochs={epochs}; max_len={MAX_LENGTH}; checkpoint={BASE_CHECKPOINT}; "
           f"config={config_hash}")

    shutil.rmtree(ckpt_dir)
    print(f"published {checkpoint_url} and removed the local copy")

    del eval_model
    gc.collect()
    torch.cuda.empty_cache()
    return scores, checkpoint_url


in_language = {}    # (seed, language) -> scores
for seed in SEEDS:
    for lang in TARGET_LANGS:
        in_language[(seed, lang)], _ = train_run(lang, TRAIN_SUBSET_SIZE, seed=seed)


## 3. Transfer matrix and gap

Rows = training language, columns = test language, showing the mean across
seeds. The **transfer gap** for a target language is `in-language F1 - zero-shot
F1`, computed per seed and then summarized by its mean, standard deviation, and
per-seed values. A gap whose sign changes across seeds is not established.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def mean_sd(values):
    values = np.array(values, dtype=float)
    return values.mean(), (values.std(ddof=1) if len(values) > 1 else 0.0)


cell_scores = {(SOURCE_LANG, lang): [zero_shot[(s, lang)] for s in SEEDS] for lang in EVAL_LANGS}
cell_scores.update({(lang, lang): [in_language[(s, lang)] for s in SEEDS] for lang in TARGET_LANGS})

matrix = pd.DataFrame(np.nan, index=EVAL_LANGS, columns=EVAL_LANGS)
spread = matrix.copy()
for (train_lang, test_lang), scores in cell_scores.items():
    matrix.loc[train_lang, test_lang], spread.loc[train_lang, test_lang] = mean_sd([s["F1_overall"] for s in scores])
matrix.index.name, matrix.columns.name = "train", "test"
spread.index.name, spread.columns.name = "train", "test"
print("mean F1_overall across seeds:")
print(matrix.round(4).to_string())
print("\nstandard deviation across seeds:")
print(spread.round(4).to_string())

print("\nmean and standard deviation by entity type:")
for (train_lang, test_lang), scores in cell_scores.items():
    parts = [f"{k} {mean_sd([s[k] for s in scores])[0]:.4f} +/- {mean_sd([s[k] for s in scores])[1]:.4f}"
             for k in ("F1_PER", "F1_ORG", "F1_LOC", "F1_overall")]
    print(f"  {train_lang} -> {test_lang}: " + " | ".join(parts))

print("\ntransfer gap (in-language - zero-shot), paired by seed:")
for lang in TARGET_LANGS:
    for k in ("F1_PER", "F1_ORG", "F1_LOC", "F1_overall"):
        gaps = [in_language[(s, lang)][k] - zero_shot[(s, lang)][k] for s in SEEDS]
        gap_mean, gap_sd = mean_sd(gaps)
        print(f"  hi -> {lang} {k}: mean {gap_mean:+.4f}, sd {gap_sd:.4f}, per seed "
              + ", ".join(f"{s}: {g:+.4f}" for s, g in zip(SEEDS, gaps)))

fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(matrix.values, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(EVAL_LANGS)), EVAL_LANGS)
ax.set_yticks(range(len(EVAL_LANGS)), EVAL_LANGS)
ax.set_xlabel("test language")
ax.set_ylabel("train language")
for i in range(len(EVAL_LANGS)):
    for j in range(len(EVAL_LANGS)):
        v = matrix.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.3f}\n\u00b1{spread.values[i, j]:.3f}", ha="center", va="center",
                    color="white" if v < 0.6 else "black")
ax.set_title(f"Phase 4: MuRIL entity F1 (mean of {len(SEEDS)} seeds)")
fig.colorbar(im, ax=ax)
fig.tight_layout()
fig_path = os.path.join(REPO_ROOT, "results", "phase4_transfer_matrix.png")
fig.savefig(fig_path, dpi=150)
print("saved", fig_path)


## Gate

```
uv run python scripts/phase4_gate.py
```

Checks that this notebook ran once from a fresh kernel; that all 15 required
rows (five matrix cells on three seeds) are logged once each from one code
commit and match its output; that the Hindi rows use the pinned Phase 3
checkpoints and are within 0.001 of their scores; that the in-language runs use the
Phase 3 size and single-GPU setup with pinned checkpoints; and that
[docs/phase4/README.md](../docs/phase4/README.md) reports each cell's mean.